## L1-NW1 低通滤波阶数对比

这个 notebook 用单井 `L1-NW1` 检查 `cup.seismic.lfm_time.lowpass_twt_log()` 中 Butterworth 低通滤波阶数的影响。

- 输入：WTIE 输出的 L1-NW1 AI 与 MD 域时深表
- 转换：先把 MD 域 AI 转到 TWT 域
- 对比：固定 `cutoff_hz = 10 Hz`，改变 `order`
- 输出：滤波曲线、相对原始 TWT AI 的残差、以及简单平滑度指标表


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from scipy.signal import butter, sosfiltfilt
from cup.petrel.load import import_checkshots_petrel, load_vp_rho_logset_from_las
from wtie.processing import grid

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

WELL_NAME = "L1-NW1"
INTERPRETATION_OFFSET = 0.03
DT_TWT = 0.002
LOWPASS_CUTOFF_HZ = 10.0
LOWPASS_ORDERS = [2, 4, 6, 8, 10]
BUFFER_SECONDS = None
BUFFER_MODE = "reflect"
WTIE_ROOT = repo_root / "data" / "output_vertical_well_wtie_20260416"


def configure_matplotlib_fonts() -> str | None:
    candidate_fonts = [
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "Source Han Sans SC",
        "PingFang SC",
        "WenQuanYi Zen Hei",
        "Arial Unicode MS",
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in candidate_fonts:
        if font_name in available_fonts:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name, *plt.rcParams.get("font.sans-serif", [])]
            plt.rcParams["axes.unicode_minus"] = False
            return font_name
    plt.rcParams["axes.unicode_minus"] = False
    return None


selected_font = configure_matplotlib_fonts()
print(f"Selected font: {selected_font}")


In [ ]:
las_file = repo_root / "data" / "vertical_well_las_target_clear" / f"{WELL_NAME}.las"
tdtable_file = WTIE_ROOT / "tdtable" / f"{WELL_NAME}_{INTERPRETATION_OFFSET}.txt"

for path in [las_file, tdtable_file]:
    if not path.exists():
        raise FileNotFoundError(path)

md_ai_log = load_vp_rho_logset_from_las(las_file).AI
td_table_md = import_checkshots_petrel(tdtable_file, depth_domain="md")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    current_twt = grid.convert_log_from_md_to_twt(md_ai_log, td_table_md, None, DT_TWT)  # type: ignore

warning_messages = [str(item.message) for item in caught]
print(f"well: {WELL_NAME}")
print(f"MD AI samples: {len(md_ai_log)}")
print(f"TWT AI samples: {len(current_twt)}")
print(f"TWT range: {float(current_twt.basis[0]):.3f} - {float(current_twt.basis[-1]):.3f} s")
if warning_messages:
    print("MD -> TWT warnings:")
    for message in warning_messages:
        print(f"- {message}")


In [ ]:
DEFAULT_FILTER_BUFFER_CYCLES = 2.0
VALID_LOCAL_BUFFER_MODES = {"reflect", "edge"}


def local_lowpass_twt_log(
    log: grid.Log,
    cutoff_hz: float = 10.0,
    order: int = 6,
    buffer_seconds: float | None = None,
    buffer_mode: str = "reflect",
    use_external_buffer: bool = True,
) -> grid.Log:
    """Notebook-local lowpass variant for comparing external buffer strategies."""
    if not log.is_twt:
        raise ValueError("local_lowpass_twt_log only supports TWT-domain logs.")
    if order % 2 != 0:
        raise ValueError("order must be even because zero-phase filtering applies the filter twice.")
    if use_external_buffer and buffer_mode not in VALID_LOCAL_BUFFER_MODES:
        raise ValueError(f"buffer_mode must be one of {sorted(VALID_LOCAL_BUFFER_MODES)}, got {buffer_mode!r}.")

    fs = 1.0 / log.sampling_rate
    nyquist = 0.5 * fs
    if cutoff_hz <= 0.0 or cutoff_hz >= nyquist:
        raise ValueError(f"cutoff_hz must be within (0, {nyquist}), got {cutoff_hz}.")

    values = log.values.astype(np.float64)
    if values.size <= 1:
        return grid.Log(values.copy(), log.basis.copy(), "twt", name=log.name, unit=getattr(log, "unit", None))

    pad_samples = 0
    values_to_filter = values
    if use_external_buffer:
        dt = float(log.sampling_rate)
        resolved_buffer_seconds = (
            max(DEFAULT_FILTER_BUFFER_CYCLES / cutoff_hz, 3.0 * order * dt)
            if buffer_seconds is None
            else float(buffer_seconds)
        )
        if resolved_buffer_seconds < 0.0:
            raise ValueError(f"buffer_seconds must be non-negative, got {buffer_seconds}.")
        pad_samples = int(np.ceil(resolved_buffer_seconds / dt))
        pad_samples = min(max(pad_samples, 0), values.size - 1)
        if pad_samples > 0:
            values_to_filter = np.pad(values, (pad_samples, pad_samples), mode=buffer_mode)

    sos = butter(order // 2, cutoff_hz, btype="low", fs=fs, output="sos")
    filtered = sosfiltfilt(sos, values_to_filter)
    if pad_samples > 0:
        filtered = filtered[pad_samples : pad_samples + values.size]

    return grid.Log(
        filtered.astype(np.float64),
        log.basis.copy(),
        "twt",
        name=log.name,
        unit=log.unit,
        allow_nan=log.allow_nan,
    )


In [ ]:
filtered_by_order = {
    order: local_lowpass_twt_log(
        current_twt,
        cutoff_hz=LOWPASS_CUTOFF_HZ,
        order=order,
        buffer_seconds=BUFFER_SECONDS,
        buffer_mode=BUFFER_MODE,
        use_external_buffer=True,
    )
    for order in LOWPASS_ORDERS
}

raw_values = current_twt.values.astype(float)
summary_records = []
for order, log in filtered_by_order.items():
    values = log.values.astype(float)
    residual = values - raw_values
    first_diff = np.diff(values)
    second_diff = np.diff(values, n=2)
    summary_records.append(
        {
            "order": order,
            "residual_rms": float(np.sqrt(np.mean(residual**2))),
            "residual_p95_abs": float(np.nanpercentile(np.abs(residual), 95)),
            "mean_abs_first_diff": float(np.mean(np.abs(first_diff))),
            "mean_abs_second_diff": float(np.mean(np.abs(second_diff))),
            "ai_min": float(np.nanmin(values)),
            "ai_max": float(np.nanmax(values)),
        }
    )

summary_df = pd.DataFrame.from_records(summary_records)
summary_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8), sharey=True, constrained_layout=True)

cmap = plt.get_cmap("viridis")
order_colors = {order: cmap(i / max(len(LOWPASS_ORDERS) - 1, 1)) for i, order in enumerate(LOWPASS_ORDERS)}

axes[0].plot(raw_values, current_twt.basis, color="#B0B0B0", lw=1.0, alpha=0.75, label="raw TWT AI")
for order, log in filtered_by_order.items():
    axes[0].plot(log.values, log.basis, lw=1.8, color=order_colors[order], label=f"order={order}")
axes[0].set_title(f"Lowpass AI curves ({LOWPASS_CUTOFF_HZ:g} Hz cutoff)")
axes[0].set_xlabel("AI")
axes[0].set_ylabel("TWT (s)")
axes[0].legend(loc="best", fontsize=9, frameon=True)

for order, log in filtered_by_order.items():
    axes[1].plot(log.values - raw_values, log.basis, lw=1.6, color=order_colors[order], label=f"order={order}")
axes[1].axvline(0.0, color="#555555", lw=0.8)
axes[1].set_title("Filtered - raw residual")
axes[1].set_xlabel("AI residual")

for ax in axes:
    ax.invert_yaxis()
    ax.grid(True, alpha=0.25)

fig.suptitle(f"{WELL_NAME}: Butterworth order comparison", fontsize=14, fontweight="bold")
plt.show()


In [ ]:
# 可按需要修改这个窗口，看局部边界、尖峰和相位差。
ZOOM_TWT_MIN = 5.0
ZOOM_TWT_MAX = 5.1

fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharey=True, constrained_layout=True)
zoom_mask = (current_twt.basis >= ZOOM_TWT_MIN) & (current_twt.basis <= ZOOM_TWT_MAX)

axes[0].plot(
    raw_values[zoom_mask], current_twt.basis[zoom_mask], color="#B0B0B0", lw=1.0, alpha=0.75, label="raw TWT AI"
)
for order, log in filtered_by_order.items():
    axes[0].plot(log.values[zoom_mask], log.basis[zoom_mask], lw=2.0, color=order_colors[order], label=f"order={order}")
axes[0].set_title("Zoomed lowpass curves")
axes[0].set_xlabel("AI")
axes[0].set_ylabel("TWT (s)")

for order, log in filtered_by_order.items():
    axes[1].plot(
        (log.values - raw_values)[zoom_mask],
        log.basis[zoom_mask],
        lw=1.8,
        color=order_colors[order],
        label=f"order={order}",
    )
axes[1].axvline(0.0, color="#555555", lw=0.8)
axes[1].set_title("Zoomed residual")
axes[1].set_xlabel("AI residual")

for ax in axes:
    ax.set_ylim(ZOOM_TWT_MAX, ZOOM_TWT_MIN)
    ax.grid(True, alpha=0.25)
axes[0].legend(loc="best", fontsize=9, frameon=True)

fig.suptitle(f"{WELL_NAME}: local view of order effect", fontsize=14, fontweight="bold")
plt.show()


In [ ]:
BUFFER_COMPARE_ORDER = 6
buffer_strategy_logs = {
    "reflect external buffer": local_lowpass_twt_log(
        current_twt,
        cutoff_hz=LOWPASS_CUTOFF_HZ,
        order=BUFFER_COMPARE_ORDER,
        buffer_seconds=BUFFER_SECONDS,
        buffer_mode="reflect",
        use_external_buffer=True,
    ),
    "edge external buffer": local_lowpass_twt_log(
        current_twt,
        cutoff_hz=LOWPASS_CUTOFF_HZ,
        order=BUFFER_COMPARE_ORDER,
        buffer_seconds=BUFFER_SECONDS,
        buffer_mode="edge",
        use_external_buffer=True,
    ),
    "no external buffer": local_lowpass_twt_log(
        current_twt,
        cutoff_hz=LOWPASS_CUTOFF_HZ,
        order=BUFFER_COMPARE_ORDER,
        use_external_buffer=False,
    ),
}

reference_values = buffer_strategy_logs["reflect external buffer"].values.astype(float)
buffer_summary_records = []
for strategy, log in buffer_strategy_logs.items():
    values = log.values.astype(float)
    residual_to_raw = values - raw_values
    delta_to_reflect = values - reference_values
    buffer_summary_records.append(
        {
            "strategy": strategy,
            "residual_to_raw_rms": float(np.sqrt(np.mean(residual_to_raw**2))),
            "delta_to_reflect_rms": float(np.sqrt(np.mean(delta_to_reflect**2))),
            "delta_to_reflect_p95_abs": float(np.nanpercentile(np.abs(delta_to_reflect), 95)),
            "start_delta_to_reflect": float(delta_to_reflect[0]),
            "end_delta_to_reflect": float(delta_to_reflect[-1]),
        }
    )

buffer_summary_df = pd.DataFrame.from_records(buffer_summary_records)
buffer_summary_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 7), sharey=True, constrained_layout=True)
strategy_colors = {
    "reflect external buffer": "#1f77b4",
    "edge external buffer": "#ff7f0e",
    "no external buffer": "#2ca02c",
}

axes[0].plot(raw_values, current_twt.basis, color="#B0B0B0", lw=1.0, alpha=0.65, label="raw TWT AI")
for strategy, log in buffer_strategy_logs.items():
    axes[0].plot(log.values, log.basis, lw=2.0, color=strategy_colors[strategy], label=strategy)
axes[0].set_title(f"Order {BUFFER_COMPARE_ORDER}: buffer strategy curves")
axes[0].set_xlabel("AI")
axes[0].set_ylabel("TWT (s)")
axes[0].legend(loc="best", fontsize=8, frameon=True)

for strategy, log in buffer_strategy_logs.items():
    axes[1].plot(log.values - raw_values, log.basis, lw=1.8, color=strategy_colors[strategy], label=strategy)
axes[1].axvline(0.0, color="#555555", lw=0.8)
axes[1].set_title("Filtered - raw residual")
axes[1].set_xlabel("AI residual")

for strategy, log in buffer_strategy_logs.items():
    axes[2].plot(
        log.values - reference_values,
        log.basis,
        lw=1.8,
        color=strategy_colors[strategy],
        label=strategy,
    )
axes[2].axvline(0.0, color="#555555", lw=0.8)
axes[2].set_title("Strategy - reflect external buffer")
axes[2].set_xlabel("AI delta")

for ax in axes:
    ax.invert_yaxis()
    ax.grid(True, alpha=0.25)

fig.suptitle(f"{WELL_NAME}: external buffer vs no external buffer", fontsize=14, fontweight="bold")
plt.show()


In [ ]:
output_dir = repo_root / "data" / "_figures"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"lfm_time_lowpass_order_compare_{WELL_NAME}@20260423.png"

fig, ax = plt.subplots(figsize=(6, 8), constrained_layout=True)
ax.plot(raw_values, current_twt.basis, color="#B0B0B0", lw=1.0, alpha=0.65, label="raw TWT AI")
for order, log in filtered_by_order.items():
    ax.plot(log.values, log.basis, lw=1.8, color=order_colors[order], label=f"order={order}")
ax.invert_yaxis()
ax.set_xlabel("AI")
ax.set_ylabel("TWT (s)")
ax.set_title(f"{WELL_NAME}: {LOWPASS_CUTOFF_HZ:g} Hz lowpass order comparison")
ax.legend(loc="best", fontsize=9, frameon=True)
ax.grid(True, alpha=0.25)
fig.savefig(output_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {output_path}")
